In [ ]:
# foodrag bootstrap: run from repo root + load .env
from foodrag.notebook import setup
setup()

In [ ]:
import logging
import nest_asyncio
from llama_index.core import Settings
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.workflow import Context
from llama_index.core.workflow import StopEvent
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.llms.litellm import LiteLLM

from foodrag.food_rag.agents import heuristic_agents
from foodrag.food_rag.agents import workflow_runner
from foodrag.food_rag.agents.heuristic_agents import FinalizationAgent
from foodrag.food_rag.agents.heuristic_agents import RecommendationAgent, ValidationAgent, SubstitutionAgent, \
    ExplanationAgent
from foodrag.food_rag.query_engines.rag import RecipeRAGQueryEngine
from foodrag.food_rag.retriever import pandas_retriever
from foodrag.food_rag.tools import finalization_tool
from foodrag.food_rag.tools import generalization_tools, ingredient_tools, validation_tools, explanation_tools, \
    meal_plan_tools, recipe_tools
from foodrag.food_rag.tools import substitution_tools
from foodrag.food_rag.tools import workflow_tools
from foodrag.food_rag.utils import embedding_helper, hummus_docify
from foodrag.food_rag.utils.recipe_context import RecipeState
from foodrag.utils import hummus_util, load_util, rdf_util
from foodrag.utils.jupyter_magic import *
from foodrag.utils.rdf_util import PreProcessedFoodOn

logging.basicConfig(
    stream=sys.stdout, level=logging.WARN
)  # logging.INFO or even logging.DEBUG for more verbose output

nest_asyncio.apply()  # Required to run the notebook in Jupyter for disabling async operations in the notebook

In [ ]:
# Output Path
data_path = './data'
embedding_path = data_path + '/embeddings/snowflake_l'
qa_name = 'small_1'

# Load configs
nutrition_config = load_util.get_config('config/nutrition_config.yml')
persona_config = load_util.get_config('config/persona_config.yml')

# Load key
api_key = load_util.load_file('config/api_key')

# 1. Setup

This notebook is based on the following llama index tutorials:
- https://docs.llamaindex.ai/en/stable/examples/agent/react_agent/
- https://docs.llamaindex.ai/en/stable/examples/agent/react_agent_with_query_engine/

### Steps:
1. Load Preprocessed Data
2. Init LLM and Sentence Transformers/Embedding

## 1.1 Load Preprocessed Data
Refer to `init_graph.ipynb` to preprocess the data. As soon as the data is preprocessed, it can be loaded here.

In [ ]:
# Load the current recipes data for the pandas retriever only
hummus_data = load_util.get_current_hummus(data_path)

# Clean df
cleaned_recipes_df = hummus_data['recipes_df'].copy()
cleaned_recipes_df = hummus_docify.stringify_nested_lists(cleaned_recipes_df)
cleaned_recipes_df = hummus_docify.stringify_nested_dicts(cleaned_recipes_df)
cleaned_recipes_df = cleaned_recipes_df.rename(
    columns={'duration': 'duration [min]', 'ingredients': 'ingredient_quantities',
             'ingredient_names': 'ingredients', 'direction_size': 'number_of_directions',
             'ingredients_sizes': 'number_of_ingredients'})
cleaned_recipes_df = cleaned_recipes_df.drop(
    columns=['last_changed_date', 'food_kg_locator', 'recipe_url', 'normalization_comment'])
cleaned_recipes_df = hummus_util.clean_serves_to_int(cleaned_recipes_df)

# Preprocess foodOn
food_on: PreProcessedFoodOn = rdf_util.PreProcessedFoodOn(hummus_data)
hummus_data['food_on'] = food_on

In [ ]:
# Load the embeddings. Note: Only a subset of the food product embeddings is loaded here for debugging purposes.
ingredient_embeddings, _ = embedding_helper.load_existing_batches(embedding_path + '/ingredient_documents')
food_product_embeddings, _ = embedding_helper.load_existing_batches(embedding_path + '/food_product_documents',
                                                                    max_batches=25, nodes_only=True)
food_on_embeddings, _ = embedding_helper.load_existing_batches(embedding_path + '/food_on_documents', nodes_only=True)

## 1.2 Init LLM and Sentence Transformers/Embedding

In [ ]:
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1"
)

# IMPORTANT NOTE: Embed model MUST be the same as the one used for the loaded embeddings
embed_model = LiteLLMEmbedding(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model_name="openai/Snowflake/snowflake-arctic-embed-l",
)
splitter, small_splitter, _ = embedding_helper.infer_splitter(embed_model)  # Splitter for the embed model

# Set Global Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = splitter.chunk_size

# 1.3 Setup RAG

In [ ]:
# Set up the retriever
retriever = pandas_retriever.PandasQueryRetriever(df=cleaned_recipes_df, verbose=False)

# Plug into RecipeRAGQueryEngine to synthesize a response
query_engine = RecipeRAGQueryEngine(retriever=retriever, llm=llm, ranking_strategy='rating', k=5, llm_calls=2,
                                    debug=True)

# 2. Recipe Tools
## 2.1 Recipe Recommendation Tool

In [ ]:
state = workflow_runner.get_dummy_state()
tool_1 = recipe_tools.RecipeRecommendationTool(recipe_query_engine=query_engine)
tool_1.recommend_recipe(ctx=state.get_context(), question="What is a recipe for hummus?")

## 2.2 Recipe Lookup Tool

In [ ]:
state = workflow_runner.get_dummy_state()
tool_2 = recipe_tools.RecipeRetrievalTool(recipe_retriever=retriever, llm_calls=1)
tool_2.retrieve_recipe(ctx=state.get_context(), title="Layered Hummus Dip")

## 2.3 Meal Plan Recommendation Tool

In [ ]:
state = workflow_runner.get_dummy_state()
tool_3_1 = meal_plan_tools.MealPlanRecommendationTool(llm=llm, recipe_query_engine=query_engine, retrieve_only=False,
                                                      time="morning",
                                                      date="13.04.2025")  # Set the time to "now" for current time- for eval a static time fits better
tool_3_1.recommend_meal_plan(ctx=state.get_context(),
                             question="What is a recipe for hummus? I also want to eat something sweet in the evening.")
state.print(keys=[RecipeState.MEAL_PLAN_QUESTIONS_KEY, RecipeState.MEAL_PLAN_RECOMMENDATION_KEY])

In [ ]:
state = workflow_runner.get_dummy_state()
tool_3_2 = meal_plan_tools.MealPlanRecommendationTool(llm=llm, recipe_query_engine=query_engine, retrieve_only=True,
                                                      time="morning",
                                                      date="13.04.2025")  # Set the time to "now" for current time- for eval a static time fits better
tool_3_2.recommend_meal_plan(ctx=state.get_context(),
                             question="What is a recipe for hummus? I also want to eat something sweet in the evening.")
state.print(keys=[RecipeState.MEAL_PLAN_QUESTIONS_KEY, RecipeState.MEAL_PLAN_RECOMMENDATION_KEY])

## 2.4 Recipe Query Generalization Tool

In [ ]:
state = workflow_runner.get_dummy_state()
tool_4 = generalization_tools.RecipeQuestionGeneralizationTool(llm=llm)
tool_4_1_response = tool_4.generalize_question(
    ctx=state.get_context(),
    question="What is a recipe for hummus with less then 20 mg sugar and no meat?", generalization_level=1)
print(tool_4_1_response)

In [ ]:
state = workflow_runner.get_dummy_state()
tool_4_2_response = tool_4.generalize_question(
    ctx=state.get_context(),
    question="What is a recipe for hummus with less then 20 mg sugar and no meat?", generalization_level=2)
print(tool_4_2_response)

In [ ]:
state = workflow_runner.get_dummy_state()
tool_4_3_response = tool_4.generalize_question(
    ctx=state.get_context(),
    question="What is a recipe for hummus with less then 20 mg sugar and no meat?", generalization_level=3)
print(tool_4_3_response)

In [ ]:
state = workflow_runner.get_dummy_state()
tool_4_4_response = tool_4.generalize_question(
    ctx=state.get_context(),
    question="What is a recipe for hummus with less then 20 mg sugar and no meat?", generalization_level=4)
print(tool_4_4_response)

In [ ]:
state = workflow_runner.get_dummy_state()
tool_4_5_response = tool_4.generalize_question(
    ctx=state.get_context(),
    question="What is a recipe for hummus with less then 20 mg sugar and no meat?", generalization_level=5)
print(tool_4_5_response)

## 2.5 Ingredient Substitution Tool

In [ ]:
tool_5_1 = ingredient_tools.IngredientSubstitutionTool(hummus_data=hummus_data, llm=llm, embed_model=embed_model,
                                                       ingredient_embeddings=ingredient_embeddings,
                                                       food_product_embeddings=food_product_embeddings,
                                                       food_on_embeddings=food_on_embeddings,
                                                       retrieving_mode=ingredient_tools.DOC_2_VEC)
tool_5_1_response = tool_5_1.find_ingredient_substitutions(ingredients=['hummus', 'banana', 'meat'])
print(tool_5_1_response)

In [ ]:
tool_5_2 = ingredient_tools.IngredientHierarchyTool(hummus_data=hummus_data, llm=llm, embed_model=embed_model,
                                                    food_on_embeddings=food_on_embeddings,
                                                    min_similarity_score=0.9,
                                                    retrieving_mode=ingredient_tools.DOC_2_VEC)
tool_5_2_response = tool_5_2.get_ingredient_hierarchy(ingredients=['hummus', 'banana', 'meat'])
print(tool_5_2_response)

## 2.6 Validation Tool

In [ ]:
state = workflow_runner.get_dummy_state()
tool_6_1_a = validation_tools.RecipeValidationTool(llm=llm, recipe_df=cleaned_recipes_df)
tool_6_1_a_response = tool_6_1_a.validate_recipe(ctx=state.get_context(),
                                                 original_question="What is a recipe for hummus?",
                                                 recipe_title="Layered Hummus Dip")
print(tool_6_1_a_response)

In [ ]:
state = workflow_runner.get_dummy_state()
ingredient_tool = ingredient_tools.IngredientHierarchyTool(hummus_data=hummus_data, llm=llm, embed_model=embed_model,
                                                           food_on_embeddings=food_on_embeddings,
                                                           min_similarity_score=0.9,
                                                           retrieving_mode=ingredient_tools.DOC_2_VEC)
tool_6_1_b = validation_tools.RecipeValidationTool(llm=llm, recipe_df=cleaned_recipes_df, use_food_on=True,
                                                   ingredient_tool=ingredient_tool)
tool_6_1_b_response = tool_6_1_b.validate_recipe(ctx=state.get_context(),
                                                 original_question="What is a recipe for hummus?",
                                                 recipe_title="Layered Hummus Dip")
print(tool_6_1_b_response)

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
})

tool_6_2_a = validation_tools.MealPlanValidationTool(llm=llm, recipe_df=cleaned_recipes_df)
tool_6_2_a.validate_meal_plan(ctx=state.get_context())
state.print(keys=[RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY, RecipeState.MEAL_PLAN_VALIDATION_KEY])

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
})
ingredient_tool = ingredient_tools.IngredientHierarchyTool(hummus_data=hummus_data, llm=llm, embed_model=embed_model,
                                                           food_on_embeddings=food_on_embeddings,
                                                           min_similarity_score=0.9,
                                                           retrieving_mode=ingredient_tools.DOC_2_VEC)
tool_6_2_b = validation_tools.MealPlanValidationTool(llm=llm, recipe_df=cleaned_recipes_df, use_food_on=True,
                                                     ingredient_tool=ingredient_tool)
tool_6_2_b.validate_meal_plan(ctx=state.get_context())
state.print(keys=[RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY, RecipeState.MEAL_PLAN_VALIDATION_KEY])

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
    RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY: "VALID \n\nThe provided meal plan is valid because it meets all the requirements specified in the question. The user asked for a recipe for hummus, which is present in both breakfast and lunch meals. Although there's repetition of the same recipe, this doesn't necessarily invalidate the meal plan since the user didn't explicitly ask for different recipes.\n\nHowever, the user also mentioned wanting to eat something sweet in the evening, but there's no dinner or dessert option provided in the meal plan that meets this requirement. Despite this, I'm considering the meal plan as valid because it partially fulfills the user's request (hummus recipe) and doesn't contain any information about dislikes or dietary restrictions.\n\nTo improve the meal plan, I would suggest adding a dinner or dessert option that includes something sweet to fulfill the user's evening craving.",
    RecipeState.MEAL_PLAN_VALIDATION_KEY: {
        "breakfast": "VALID \n\nThis recipe is valid because it contains hummus as a main ingredient. The question asks for a recipe for hummus, and this recipe features hummus prominently, combining it with cumin and using it as the base layer of the dip. Although the recipe doesn't provide a method for making hummus from scratch (it uses prepared hummus), it still meets the requirement by including hummus as a key component.\n\nSince there are no specified whitelist or blacklist requirements in the question beyond asking for a hummus recipe, this dish qualifies based on its primary ingredient alone.",
        "lunch": "INVALID\n\nThe recipe \"Layered Hummus Dip\" is invalid based on the user's question because there is no indication that this recipe is sweet or suitable for an evening dessert. The ingredients listed are savory, not sweet, and the description does not mention anything about being a sweet treat.\n\nTo fix this, you could suggest adding a sweet ingredient to the recipe, such as honey or fruit, or recommend a different recipe that is specifically designed to be a sweet treat in the evening."
    },
})

tool_6_3_a = validation_tools.CorrectionTool(llm=llm, recipe_df=cleaned_recipes_df, recipe_query_engine=query_engine)
tool_6_3_a.correct_recipes(ctx=state.get_context())
state.print()

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
    RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY: "VALID \n\nThe provided meal plan is valid because it meets all the requirements specified in the question. The user asked for a recipe for hummus, which is present in both breakfast and lunch meals. Although there's repetition of the same recipe, this doesn't necessarily invalidate the meal plan since the user didn't explicitly ask for different recipes.\n\nHowever, the user also mentioned wanting to eat something sweet in the evening, but there's no dinner or dessert option provided in the meal plan that meets this requirement. Despite this, I'm considering the meal plan as valid because it partially fulfills the user's request (hummus recipe) and doesn't contain any information about dislikes or dietary restrictions.\n\nTo improve the meal plan, I would suggest adding a dinner or dessert option that includes something sweet to fulfill the user's evening craving.",
    RecipeState.MEAL_PLAN_VALIDATION_KEY: {
        "breakfast": "VALID \n\nThis recipe is valid because it contains hummus as a main ingredient. The question asks for a recipe for hummus, and this recipe features hummus prominently, combining it with cumin and using it as the base layer of the dip. Although the recipe doesn't provide a method for making hummus from scratch (it uses prepared hummus), it still meets the requirement by including hummus as a key component.\n\nSince there are no specified whitelist or blacklist requirements in the question beyond asking for a hummus recipe, this dish qualifies based on its primary ingredient alone.",
        "lunch": "INVALID\n\nThe recipe \"Layered Hummus Dip\" is invalid based on the user's question because there is no indication that this recipe is sweet or suitable for an evening dessert. The ingredients listed are savory, not sweet, and the description does not mention anything about being a sweet treat.\n\nTo fix this, you could suggest adding a sweet ingredient to the recipe, such as honey or fruit, or recommend a different recipe that is specifically designed to be a sweet treat in the evening."
    },
})
ingredient_tool = ingredient_tools.IngredientHierarchyTool(hummus_data=hummus_data, llm=llm, embed_model=embed_model,
                                                           food_on_embeddings=food_on_embeddings,
                                                           min_similarity_score=0.9,
                                                           retrieving_mode=ingredient_tools.DOC_2_VEC)
tool_6_3_b = validation_tools.CorrectionTool(llm=llm, recipe_df=cleaned_recipes_df, recipe_query_engine=query_engine,
                                             use_food_on=True, ingredient_tool=ingredient_tool)
tool_6_3_b.correct_recipes(ctx=state.get_context())
state.print()

## 2.7 Substitution Tool

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
    RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY: "VALID \n\nThe provided meal plan is valid because it meets all the requirements specified in the question. The user asked for a recipe for hummus, which is present in both breakfast and lunch meals. Although there's repetition of the same recipe, this doesn't necessarily invalidate the meal plan since the user didn't explicitly ask for different recipes.\n\nHowever, the user also mentioned wanting to eat something sweet in the evening, but there's no dinner or dessert option provided in the meal plan that meets this requirement. Despite this, I'm considering the meal plan as valid because it partially fulfills the user's request (hummus recipe) and doesn't contain any information about dislikes or dietary restrictions.\n\nTo improve the meal plan, I would suggest adding a dinner or dessert option that includes something sweet to fulfill the user's evening craving.",
    RecipeState.MEAL_PLAN_VALIDATION_KEY: {
        "breakfast": "VALID \n\nThis recipe is valid because it contains hummus as a main ingredient. The question asks for a recipe for hummus, and this recipe features hummus prominently, combining it with cumin and using it as the base layer of the dip. Although the recipe doesn't provide a method for making hummus from scratch (it uses prepared hummus), it still meets the requirement by including hummus as a key component.\n\nSince there are no specified whitelist or blacklist requirements in the question beyond asking for a hummus recipe, this dish qualifies based on its primary ingredient alone.",
        "lunch": "INVALID\n\nThe recipe \"Layered Hummus Dip\" is invalid based on the user's question because there is no indication that this recipe is sweet or suitable for an evening dessert. The ingredients listed are savory, not sweet, and the description does not mention anything about being a sweet treat.\n\nTo fix this, you could suggest adding a sweet ingredient to the recipe, such as honey or fruit, or recommend a different recipe that is specifically designed to be a sweet treat in the evening."
    },
    RecipeState.HISTORY_KEY: [
        "Reformulated the question 'What is a recipe for hummus? I also want to eat something sweet in the evening.' into meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}",
        "Based on those meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}\n recommended this meal plan: 'breakfast': 'Layered Hummus Dip', 'lunch': 'Linda's Thai  Sweet Chili  Sauce for Dipping (Egg Rolls, Sushi)'"
    ]
})

tool_8 = substitution_tools.SubstitutionTool(llm=llm)
tool_8.substitute(ctx=state.get_context(), objectives=["easy to cook", "not expensive"])
state.print([RecipeState.SUBSTITUTED_MEAL_PLAN_RECOMMENDATION_KEY])

## 2.8 Explanation Generation Tool

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
    RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY: "VALID \n\nThe provided meal plan is valid because it meets all the requirements specified in the question. The user asked for a recipe for hummus, which is present in both breakfast and lunch meals. Although there's repetition of the same recipe, this doesn't necessarily invalidate the meal plan since the user didn't explicitly ask for different recipes.\n\nHowever, the user also mentioned wanting to eat something sweet in the evening, but there's no dinner or dessert option provided in the meal plan that meets this requirement. Despite this, I'm considering the meal plan as valid because it partially fulfills the user's request (hummus recipe) and doesn't contain any information about dislikes or dietary restrictions.\n\nTo improve the meal plan, I would suggest adding a dinner or dessert option that includes something sweet to fulfill the user's evening craving.",
    RecipeState.MEAL_PLAN_VALIDATION_KEY: {
        "breakfast": "VALID \n\nThis recipe is valid because it contains hummus as a main ingredient. The question asks for a recipe for hummus, and this recipe features hummus prominently, combining it with cumin and using it as the base layer of the dip. Although the recipe doesn't provide a method for making hummus from scratch (it uses prepared hummus), it still meets the requirement by including hummus as a key component.\n\nSince there are no specified whitelist or blacklist requirements in the question beyond asking for a hummus recipe, this dish qualifies based on its primary ingredient alone.",
        "lunch": "INVALID\n\nThe recipe \"Layered Hummus Dip\" is invalid based on the user's question because there is no indication that this recipe is sweet or suitable for an evening dessert. The ingredients listed are savory, not sweet, and the description does not mention anything about being a sweet treat.\n\nTo fix this, you could suggest adding a sweet ingredient to the recipe, such as honey or fruit, or recommend a different recipe that is specifically designed to be a sweet treat in the evening."
    },
    RecipeState.SUBSTITUTED_MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": "{\n  \"title\": \"Simplified Layered Hummus Dip\",\n  \"directions\": [\n    \"1. Combine hummus and cumin in the bottom of a 9 inch pie dish.\",\n    \"2. Top with a layer of tomatoes, seasoned with garlic salt.\",\n    \"3. Add the cucumbers and sprinkle with pepper.\",\n    \"4. Add a layer of feta topped with olives.\"\n  ],\n  \"ingredients\": {\n    \"hummus\": \"8 ounces\",\n    \"cumin\": \"0.125 teaspoon\",\n    \"tomatoes\": \"0.5 cup, chopped\",\n    \"garlic salt\": \"\",\n    \"cucumbers\": \"0.25 cup, chopped\",\n    \"fresh ground pepper\": \"\",\n    \"feta cheese\": \"4 ounces, crumbled\",\n    \"kalamata olives\": \"2 tablespoons, chopped\"\n  }\n}",
        "lunch": "{\n  \"title\": \"Layered Hummus Dip\",\n  \"directions\": [\n    \"Combine hummus and cumin, spread on the bottom of a 9 inch pie dish.\",\n    \"Top with a layer of tomatoes, seasoned with garlic salt.\",\n    \"Add the cucumbers and sprinkle with pepper.\",\n    \"Add a layer of feta topped with olives, then parsley.\"\n  ],\n  \"ingredients\": {\n    \"hummus\": \"8 ounces\",\n    \"cumin\": \"0.125 teaspoon\",\n    \"tomatoes\": \"0.5 cup chopped\",\n    \"garlic salt\": \"\",\n    \"cucumbers\": \"0.25 cup chopped\",\n    \"fresh ground pepper\": \"\",\n    \"feta cheese\": \"4 ounces crumbled\",\n    \"kalamata olives\": \"2 tablespoons chopped\",\n    \"parsley\": \"2 tablespoons chopped\"\n  }\n}"
    },
    RecipeState.HISTORY_KEY: [
        "Reformulated the question 'What is a recipe for hummus? I also want to eat something sweet in the evening.' into meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}",
        "Based on those meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}\n recommended this meal plan: 'breakfast': 'Layered Hummus Dip', 'lunch': 'Linda's Thai  Sweet Chili  Sauce for Dipping (Egg Rolls, Sushi)'"
    ]
})

tool_8 = explanation_tools.ExplanationTool(llm=llm, hummus_data=hummus_data)
tool_8.explain(ctx=state.get_context())
state.print([RecipeState.MEAL_PLAN_EXPLANATION_KEY])

## 2.9 Finalization Tool

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
    RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY: "VALID \n\nThe provided meal plan is valid because it meets all the requirements specified in the question. The user asked for a recipe for hummus, which is present in both breakfast and lunch meals. Although there's repetition of the same recipe, this doesn't necessarily invalidate the meal plan since the user didn't explicitly ask for different recipes.\n\nHowever, the user also mentioned wanting to eat something sweet in the evening, but there's no dinner or dessert option provided in the meal plan that meets this requirement. Despite this, I'm considering the meal plan as valid because it partially fulfills the user's request (hummus recipe) and doesn't contain any information about dislikes or dietary restrictions.\n\nTo improve the meal plan, I would suggest adding a dinner or dessert option that includes something sweet to fulfill the user's evening craving.",
    RecipeState.MEAL_PLAN_VALIDATION_KEY: {
        "breakfast": "VALID \n\nThis recipe is valid because it contains hummus as a main ingredient. The question asks for a recipe for hummus, and this recipe features hummus prominently, combining it with cumin and using it as the base layer of the dip. Although the recipe doesn't provide a method for making hummus from scratch (it uses prepared hummus), it still meets the requirement by including hummus as a key component.\n\nSince there are no specified whitelist or blacklist requirements in the question beyond asking for a hummus recipe, this dish qualifies based on its primary ingredient alone.",
        "lunch": "INVALID\n\nThe recipe \"Layered Hummus Dip\" is invalid based on the user's question because there is no indication that this recipe is sweet or suitable for an evening dessert. The ingredients listed are savory, not sweet, and the description does not mention anything about being a sweet treat.\n\nTo fix this, you could suggest adding a sweet ingredient to the recipe, such as honey or fruit, or recommend a different recipe that is specifically designed to be a sweet treat in the evening."
    },
    RecipeState.HISTORY_KEY: [
        "Reformulated the question 'What is a recipe for hummus? I also want to eat something sweet in the evening.' into meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}",
        "Based on those meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}\n recommended this meal plan: 'breakfast': 'Layered Hummus Dip', 'lunch': 'Linda's Thai  Sweet Chili  Sauce for Dipping (Egg Rolls, Sushi)'"
    ],
    RecipeState.MEAL_PLAN_EXPLANATION_KEY: "The recommended meal plan is tailored to meet your specific requirements. For breakfast, the \"Layered Hummus Dip\" recipe is suggested as it directly addresses your question about a hummus recipe. This dish is not only delicious but also provides a Mediterranean twist on a popular dip, incorporating hummus with cumin, tomatoes, cucumbers, feta cheese, olives, and parsley.\n\nFor lunch, the recommendation was initially \"Linda's Thai Sweet Chili Sauce for Dipping (Egg Rolls, Sushi),\" which aimed to satisfy your desire for something sweet in the evening. However, this dish doesn't directly fit into a typical lunch category or the provided meal plan structure. Given the context and focusing on the primary goal of addressing your hummus query while also considering a sweet element, it's clear that the initial approach was to offer a versatile dip that could be enjoyed at various times.\n\nThe \"Layered Hummus Dip\" stands out as a versatile option that can be enjoyed not just for breakfast but potentially as a snack or light meal at any time of day. Its flavors and ingredients provide a satisfying experience that aligns with your interest in hummus, even though it may not directly fulfill the sweet craving mentioned for the evening.\n\nIn summary, the recommended \"Layered Hummus Dip\" is a thoughtful response to your query about hummus, offering a unique and flavorful way to enjoy this popular dip. While the initial plan attempted to address both aspects of your question, the focus on providing a high-quality hummus recipe takes precedence, with the understanding that sweet cravings can be met through other means or at different times."
})
tool_9_1 = finalization_tool.FinalizeRecommendationTool(llm=llm, mode=finalization_tool.MODE_SIMPLE)
tool_9_1.finalize(ctx=state.get_context())

In [ ]:
state = workflow_runner.get_dummy_state(state_data={
    RecipeState.ORIGINAL_USER_QUESTION_KEY: "What is a recipe for hummus? I also want to eat something sweet in the evening.",
    RecipeState.MEAL_PLAN_QUESTIONS_KEY: {
        "breakfast": "What is a recipe for hummus?",
        "lunch": "I also want to eat something sweet in the evening."
    },
    RecipeState.MEAL_PLAN_RECOMMENDATION_KEY: {
        "breakfast": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2,
                      'caloriesFromFat [kcal]': 229.0, 'cholesterol [mg]': 53.6,
                      'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                      'dietaryFiber [g]': 8.3,
                      'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                      'duration [min]': 15.0, 'fsa_score': 0.25,
                      'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                      'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                      'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                      'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                      'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                      'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                      'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                      'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                      'who_score': 0.2325207860922147},
        "lunch": {'author_id': 1858, 'average_rating': 4.93, 'calories [kcal]': 382.2, 'caloriesFromFat [kcal]': 229.0,
                  'cholesterol [mg]': 53.6,
                  'description': 'The Mediterranean version of the ever popular Mexican layered dip. Instead of using garlic salt, I mix a large clove of pressed garlic with the hummus. Serve with toasted pita, flat bread or tortilla chips.',
                  'dietaryFiber [g]': 8.3,
                  'directions': 'Combine hummus and cumin, spread on the bottom of a 9 inch pie dish. Top with a layer of tomatoes, seasoned with garlic salt. Add the cucumbers and sprinkle with pepper. Add a layer of feta topped with olives, then parsley.',
                  'duration [min]': 15.0, 'fsa_score': 0.25,
                  'ingredient_quantities': '8 ounces of prepared hummus; 0.125 teaspoon of cumin; 0.5 cup of chopped tomato; garlic salt; 0.25 cup of chopped cucumber; fresh ground pepper; 4 ounces of crumbled feta cheese; 2 tablespoons of chopped kalamata olives; 2 tablespoons of chopped parsley',
                  'ingredients': 'black olives, chopped tomatoes, cucumbers, cumin, feta cheese, fresh ground pepper, garlic salt, hummus,',
                  'meal_types': 'lunch, dinner, breakfast, snack,', 'number_of_directions': 4,
                  'number_of_ingredients': 9, 'number_of_ratings': 42, 'nutri_score': 0.0, 'protein [g]': 18.8,
                  'recipe_id': 7362, 'saturatedFat [g]': 10.9, 'serves': 2.0, 'servingSize [g]': 504.0,
                  'servingsPerRecipe': 1, 'sodium [mg]': 1199.0, 'sugars [g]': 3.9,
                  'tags': 'low-in-something, vegetables, to-go, easy, comfort-food, appetizers, technique, 15-minutes-or-less, kid-friendly, beginner-cook, taste-mood, low-carb, eggs, picnic, dips, no-cook, inexpensive, dinner-party, cheese, occasion, preparation, time-to-make, cuisine, course, dietary, european, chick-peas-garbanzos, eggs-dairy, potluck, main-ingredient, beans, number-of-servings,',
                  'title': 'Layered Hummus Dip', 'totalCarbohydrate [g]': 22.9, 'totalFat [g]': 25.5,
                  'who_score': 0.2325207860922147}
    },
    RecipeState.OVERALL_MEAL_PLAN_VALIDATION_KEY: "VALID \n\nThe provided meal plan is valid because it meets all the requirements specified in the question. The user asked for a recipe for hummus, which is present in both breakfast and lunch meals. Although there's repetition of the same recipe, this doesn't necessarily invalidate the meal plan since the user didn't explicitly ask for different recipes.\n\nHowever, the user also mentioned wanting to eat something sweet in the evening, but there's no dinner or dessert option provided in the meal plan that meets this requirement. Despite this, I'm considering the meal plan as valid because it partially fulfills the user's request (hummus recipe) and doesn't contain any information about dislikes or dietary restrictions.\n\nTo improve the meal plan, I would suggest adding a dinner or dessert option that includes something sweet to fulfill the user's evening craving.",
    RecipeState.MEAL_PLAN_VALIDATION_KEY: {
        "breakfast": "VALID \n\nThis recipe is valid because it contains hummus as a main ingredient. The question asks for a recipe for hummus, and this recipe features hummus prominently, combining it with cumin and using it as the base layer of the dip. Although the recipe doesn't provide a method for making hummus from scratch (it uses prepared hummus), it still meets the requirement by including hummus as a key component.\n\nSince there are no specified whitelist or blacklist requirements in the question beyond asking for a hummus recipe, this dish qualifies based on its primary ingredient alone.",
        "lunch": "INVALID\n\nThe recipe \"Layered Hummus Dip\" is invalid based on the user's question because there is no indication that this recipe is sweet or suitable for an evening dessert. The ingredients listed are savory, not sweet, and the description does not mention anything about being a sweet treat.\n\nTo fix this, you could suggest adding a sweet ingredient to the recipe, such as honey or fruit, or recommend a different recipe that is specifically designed to be a sweet treat in the evening."
    },
    RecipeState.HISTORY_KEY: [
        "Reformulated the question 'What is a recipe for hummus? I also want to eat something sweet in the evening.' into meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}",
        "Based on those meal plan questions:\n{'breakfast': 'What is a recipe for hummus?', 'lunch': 'I also want to eat something sweet in the evening.'}\n recommended this meal plan: 'breakfast': 'Layered Hummus Dip', 'lunch': 'Linda's Thai  Sweet Chili  Sauce for Dipping (Egg Rolls, Sushi)'"
    ],
    RecipeState.MEAL_PLAN_EXPLANATION_KEY: "The recommended meal plan is tailored to meet your specific requirements. For breakfast, the \"Layered Hummus Dip\" recipe is suggested as it directly addresses your question about a hummus recipe. This dish is not only delicious but also provides a Mediterranean twist on a popular dip, incorporating hummus with cumin, tomatoes, cucumbers, feta cheese, olives, and parsley.\n\nFor lunch, the recommendation was initially \"Linda's Thai Sweet Chili Sauce for Dipping (Egg Rolls, Sushi),\" which aimed to satisfy your desire for something sweet in the evening. However, this dish doesn't directly fit into a typical lunch category or the provided meal plan structure. Given the context and focusing on the primary goal of addressing your hummus query while also considering a sweet element, it's clear that the initial approach was to offer a versatile dip that could be enjoyed at various times.\n\nThe \"Layered Hummus Dip\" stands out as a versatile option that can be enjoyed not just for breakfast but potentially as a snack or light meal at any time of day. Its flavors and ingredients provide a satisfying experience that aligns with your interest in hummus, even though it may not directly fulfill the sweet craving mentioned for the evening.\n\nIn summary, the recommended \"Layered Hummus Dip\" is a thoughtful response to your query about hummus, offering a unique and flavorful way to enjoy this popular dip. While the initial plan attempted to address both aspects of your question, the focus on providing a high-quality hummus recipe takes precedence, with the understanding that sweet cravings can be met through other means or at different times."
})
tool_9_2 = finalization_tool.FinalizeRecommendationTool(llm=llm, mode=finalization_tool.MODE_LLM_REFINED)
tool_9_2.finalize(ctx=state.get_context())

## 2.10 Workflow Tool

In [ ]:
state = workflow_runner.get_dummy_state()
tool_10 = workflow_tools.MealPlanWorkflowTool(hummus_data=hummus_data, recipe_df=cleaned_recipes_df, llm=llm,
                                              recipe_query_engine=query_engine)
tool_10.run_meal_plan_workflow(ctx=state.get_context(),
                               question="What is a recipe for hummus? I also want to eat something sweet in the evening.")

# 3. Agents
Integrate tools into agents.

## 3.1 Heuristic ReActAgent

In [ ]:
# Create the agent # TODO DOES NOT WORK RIGHT NOW
heuristic_agent = heuristic_agents.HeuristicMealPlanReActAgent(hummus_data=hummus_data,
                                                               cleaned_recipes_df=cleaned_recipes_df,
                                                               retriever=retriever, query_engine=query_engine,
                                                               embed_model=embed_model, llm=llm,
                                                               ingredient_embeddings=ingredient_embeddings,
                                                               food_product_embeddings=food_product_embeddings,
                                                               food_on_embeddings=food_on_embeddings,
                                                               max_iterations=20)
ctx = Context(heuristic_agent)

In [ ]:
# TODO --> how to handle the max iterations? Especially when the agent is evaluated later on
agent_workflow = AgentWorkflow(
    agents=[heuristic_agent],
    root_agent=heuristic_agent.name,
    initial_state=RecipeState.get_init_state()
)

handler = agent_workflow.run("What is a recipe for hummus? I also want to eat something sweet in the evening.", ctx=ctx)

async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler

In [ ]:
# Chat with agent
#human_loop_chat.chat_repl(agent)

## 3.2 Heuristic MultiAgent

In [ ]:
# Create the agents
recommendation_agent = RecommendationAgent(query_engine=query_engine)
validation_agent = ValidationAgent(llm=llm, query_engine=query_engine, recipe_df=cleaned_recipes_df)
substitution_agent = SubstitutionAgent(hummus_data=hummus_data, llm=llm)
explanation_agent = ExplanationAgent(llm=llm, hummus_data=hummus_data)
finalization_agent = FinalizationAgent(llm=llm, mode=finalization_tool.MODE_SIMPLE)

In [ ]:
# Put agents into a workflow
agent_workflow = AgentWorkflow(
    agents=[recommendation_agent, validation_agent, substitution_agent, explanation_agent, finalization_agent],
    root_agent=recommendation_agent.name,
    initial_state=RecipeState.get_init_state(),
)

In [ ]:
# Run the workflow
handler = agent_workflow.run(
    user_msg="What is a recipe for hummus for now? Also, I want to eat something sweet in the evening afterwards.",
)

current_agent = None
current_tool_calls = ""
async for event in handler.stream_events():
    if (
            hasattr(event, "current_agent_name")
            and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'=' * 50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'=' * 50}\n")

    if isinstance(event, AgentStream):
        if event.delta:
            print(event.delta, end="", flush=True)
    elif isinstance(event, AgentInput):
        print("📥 Input:", event.input)
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")
    elif isinstance(event, StopEvent):
        print("🛑 Stop Event")
    else:
        print(event)

In [ ]:
state = handler.ctx.get("state")
print(state[0][RecipeState.FINAL_RECOMMENDATION_KEY])

In [ ]:
state = RecipeState.from_context(ctx)

# 3.3 Heuristic Tool Agent

In [ ]:
tool = workflow_tools.MealPlanWorkflowTool(hummus_data=hummus_data, recipe_df=cleaned_recipes_df, llm=llm,
                                           recipe_query_engine=query_engine)
agent = heuristic_agents.HeuristicMealPlanReActAgent(hummus_data=hummus_data, cleaned_recipes_df=cleaned_recipes_df,
                                                     retriever=retriever, query_engine=query_engine,
                                                     embed_model=embed_model, llm=llm,
                                                     ingredient_embeddings=ingredient_embeddings,
                                                     food_product_embeddings=food_product_embeddings,
                                                     food_on_embeddings=food_on_embeddings,
                                                     tools=[tool.to_tool()])
agent_workflow = AgentWorkflow(
    agents=[agent],
    root_agent=agent.name,
    initial_state=RecipeState.get_init_state(),
)

In [ ]:
handler = agent_workflow.run("What is a recipe for hummus? I also want to eat something sweet in the evening.", ctx=ctx)

async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler